# Deploy — Tenant Extract (Track B orchestrator)

> **[Unverified — requires Fabric tenant pilot]**  ·  **Label: [NET-NEW]**
> Chains **`tenant_doctor` → `00_tenant_extract` → `01a_tenant_silver_gold` → `04_capacity_bridge`**
> to populate the Nexus report's tenant pages (`gold_inventory`, `gold_activities`,
> `gold_refreshables`, `gold_capacities`). This is **Track B** of `docs/RUNBOOK-F2-pilot.md`.
>
> **Before you run:** the preflight cell must print `VERDICT: READY`. If it says `BLOCKED`,
> apply the remediation it prints (usually the read-only-admin-API tenant setting) and re-run.
>
> **MOCK mode** (`TENANT_EXTRACT_MOCK=1`) runs the whole chain with synthetic data and no
> network — use it to rehearse. Live runs need an SP with admin read scopes.

In [ ]:
# =============================================================================
# (a) CONFIG + CHAIN HELPER  |  Label: [NET-NEW]
# [Unverified — not executed in live Fabric]
# =============================================================================
import os, json, importlib.util

CONFIG_PATH = os.getenv("CONFIG_PATH", "../config/config.json")
NB_DIR = os.getenv("NB_DIR", "../notebooks")

def load_config():
    if os.path.exists(CONFIG_PATH):
        cfg = json.load(open(CONFIG_PATH))
    else:
        print(f"[warn] {CONFIG_PATH} not found — running with empty config (MOCK only).")
        cfg = {}
    # Resolve SP secret from Key Vault at runtime (never store the secret in config.json).
    kv = cfg.get("keyVault") or {}
    if kv.get("vaultUri") and kv.get("spClientSecretSecretName") and not os.getenv("TENANT_EXTRACT_MOCK"):
        try:
            from azure.identity import DefaultAzureCredential
            from azure.keyvault.secrets import SecretClient
            sc = SecretClient(vault_url=kv["vaultUri"], credential=DefaultAzureCredential())
            cfg["clientSecret"] = sc.get_secret(kv["spClientSecretSecretName"]).value
        except Exception as e:  # noqa: BLE001
            print(f"[warn] Key Vault secret fetch failed ({e}); set clientSecret another way for live runs.")
    return cfg

def run_notebook(name):
    """Import + return a sibling medallion notebook module (00_tenant_extract, etc.).
    In Fabric prefer:  %run {name}   (uncomment the magic below). Off-cluster we importlib."""
    # In Fabric, replace the importlib block with:  %run ../notebooks/{name}
    path = os.path.join(NB_DIR, f"{name}.py")
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

CONFIG = load_config()
print("[config] loaded keys:", sorted(k for k in CONFIG if not k.startswith("_")))


## (b) Preflight — `tenant_doctor` (GATE)
Do not proceed unless this prints `VERDICT: READY`.

In [ ]:
# =============================================================================
# (b) PREFLIGHT — tenant_doctor  |  Label: [NET-NEW]
# =============================================================================
doctor = run_notebook("tenant_doctor")
_verdict = doctor.run(CONFIG)
assert _verdict["verdict"] == "READY", (
    f"Preflight BLOCKED ({_verdict['required_fail']} required check(s) failing). "
    "Fix the items printed above (usually the read-only-admin-API tenant setting), then re-run."
)
print("Preflight READY — proceeding to extract.")


## (c) Extract → bronze — `00_tenant_extract`
Scanner API + Activity Events + Refreshables.

In [ ]:
# =============================================================================
# (c) EXTRACT — 00_tenant_extract  |  Label: [NET-NEW]
# =============================================================================
# Ensure the three tenant collectors are enabled for THIS run. config.sample.json
# ships them off by default (opt-in); running this deploy notebook IS the opt-in,
# so force them on for the extract. Set EXTRACT_RESPECT_CONFIG_FLAGS=1 to instead
# honor whatever is in config.json.
_feats = dict(CONFIG.get("features") or {})
if not os.getenv("EXTRACT_RESPECT_CONFIG_FLAGS"):
    for _f in ("collectInventory", "collectActivityEvents", "collectRefreshables"):
        _feats[_f] = True
CONFIG["features"] = _feats

extract = run_notebook("00_tenant_extract")
bronze_counts = extract.run(CONFIG)
print("[extract] bronze counts:", bronze_counts)
assert sum(bronze_counts.values()) > 0, (
    "Extract produced 0 bronze rows. Enable features.collectInventory/collectActivityEvents/"
    "collectRefreshables, or check the SP has data to read (live) — see tenant_doctor D2/D3."
)


## (d) Conform → gold — `01a_tenant_silver_gold`
gold_inventory / gold_activities / gold_refreshables.

In [ ]:
# =============================================================================
# (d) SILVER/GOLD — 01a_tenant_silver_gold  |  Label: [NET-NEW]
# =============================================================================
tsg = run_notebook("01a_tenant_silver_gold")
gold_counts = tsg.run()
print("[silver/gold] gold counts:", gold_counts)


## (e) Capacity CU bridge — `04_capacity_bridge`
Populates `gold_capacities` + the CU measures. Set `config.capacityBridge.mode` to
`fpm_eventhouse` or `capacity_metrics_xmla` for live CU; `mock` (default) is synthetic.

In [ ]:
# =============================================================================
# (e) CAPACITY BRIDGE — 04_capacity_bridge  |  Label: [NET-NEW]
# =============================================================================
bridge = run_notebook("04_capacity_bridge")
cap_rows = bridge.run(CONFIG)
print("[capacity bridge] gold_capacities rows:", cap_rows)


## (f) Done — verify + next steps

Confirm four gold tables now have rows: `gold_inventory`, `gold_activities`,
`gold_refreshables`, `gold_capacities`. Open the **Nexus Gateway & Fabric Observatory**
report — Tenant Overview / Timeline / Refresh Analytics should populate.

Report back per **Track B** in [`docs/RUNBOOK-F2-pilot.md`](../../docs/RUNBOOK-F2-pilot.md):
the `tenant_doctor` verdict, the four row counts, and which `capacityBridge.mode` you used.

In [ ]:
# =============================================================================
# (f) SUMMARY  |  Label: [NET-NEW]
# =============================================================================
print("Tenant extract chain complete.")
print("  bronze:", bronze_counts)
print("  gold:  ", gold_counts)
print("  gold_capacities rows:", cap_rows)
